In [1]:
!pip install openpyxl

In [2]:
import os

os.makedirs("spreadsheet-automator/src", exist_ok=True)
os.makedirs("spreadsheet-automator/data", exist_ok=True)

with open("spreadsheet-automator/src/__init__.py", "w") as f:
    f.write("# This directory is a Python package.\n")

print("สร้างโครงสร้างโปรเจกต์เรียบร้อย")

สร้างโครงสร้างโปรเจกต์เรียบร้อย


In [3]:
from openpyxl import Workbook

input_path = "spreadsheet-automator/data/input_sales.xlsx"

wb = Workbook()
ws = wb.active
ws.title = "Sheet1"

ws.append(["Product Name", "Quantity", "Unit Price"])
ws.append(["Laptop", 2, 1200.50])
ws.append(["Mouse", 5, 25.00])
ws.append(["Keyboard", 3, 75.99])
ws.append(["Monitor", 1, 300.00])
ws.append(["Printer", 1, 150.00])
ws.append([None, None, None])  # แถวว่างสำหรับทดสอบ

wb.save(input_path)

print(f"สร้างไฟล์ตัวอย่างแล้ว: {input_path}")

สร้างไฟล์ตัวอย่างแล้ว: spreadsheet-automator/data/input_sales.xlsx


In [4]:
from openpyxl import load_workbook

wb = load_workbook(input_path, data_only=True)
ws = wb.active

for row in ws.iter_rows(values_only=True):
    print(row)

wb.close()

('Product Name', 'Quantity', 'Unit Price')
('Laptop', 2, 1200.5)
('Mouse', 5, 25)
('Keyboard', 3, 75.99)
('Monitor', 1, 300)
('Printer', 1, 150)


In [5]:
%%writefile spreadsheet-automator/src/excel_processor.py

import os
import openpyxl

from openpyxl.styles import (
    Font, PatternFill, Border, Side, Alignment
)
from openpyxl.utils import get_column_letter


class ExcelProcessor:
    """จัดการข้อมูลยอดขายด้วย openpyxl"""

    def create_workbook(self, filename="new_workbook.xlsx"):
        wb = openpyxl.Workbook()

        if "Sheet" in wb.sheetnames:
            del wb["Sheet"]

        print(f"สร้าง Workbook สำหรับ {filename}")
        return wb

    def load_workbook(self, filename):
        try:
            wb = openpyxl.load_workbook(filename, data_only=True)
            print(f"อ่านไฟล์ {filename} สำเร็จ")
            return wb
        except FileNotFoundError:
            print(f"ไม่พบไฟล์: {filename}")
            return None
        except Exception as e:
            print(f"เกิดข้อผิดพลาดขณะอ่านไฟล์: {e}")
            return None

    def save_workbook(self, workbook, filename):
        try:
            os.makedirs(os.path.dirname(filename) or ".", exist_ok=True)
            workbook.save(filename)
            print(f"บันทึกไฟล์แล้ว: {filename}")
            return True
        except Exception as e:
            print(f"เกิดข้อผิดพลาดขณะบันทึกไฟล์: {e}")
            return False

    def process_sales_data(self, input_filename, output_filename):
        input_wb = self.load_workbook(input_filename)

        if input_wb is None:
            return False

        try:
            input_sheet = input_wb.active

            # ตรวจสอบหัวตาราง
            expected_headers = [
                "Product Name", "Quantity", "Unit Price"
            ]
            actual_headers = [
                input_sheet.cell(row=1, column=i).value
                for i in range(1, 4)
            ]

            if actual_headers != expected_headers:
                print("หัวตารางไม่ถูกต้อง:", actual_headers)
                return False

            output_wb = self.create_workbook(output_filename)
            output_sheet = output_wb.create_sheet(
                "Sales Report", 0
            )

            headers = [
                "Product Name",
                "Quantity",
                "Unit Price",
                "Total Price"
            ]

            header_font = Font(bold=True, color="FFFFFF")
            header_fill = PatternFill(
                fill_type="solid",
                fgColor="4CAF50"
            )

            total_font = Font(bold=True, color="000000")
            total_fill = PatternFill(
                fill_type="solid",
                fgColor="FFEB3B"
            )

            thin_side = Side(style="thin", color="000000")
            thin_border = Border(
                left=thin_side,
                right=thin_side,
                top=thin_side,
                bottom=thin_side
            )

            currency_format = '#,##0.00'

            # สร้างหัวตาราง
            for col_num, header in enumerate(headers, start=1):
                cell = output_sheet.cell(
                    row=1, column=col_num, value=header
                )
                cell.font = header_font
                cell.fill = header_fill
                cell.border = thin_border
                cell.alignment = Alignment(
                    horizontal="center",
                    vertical="center"
                )

            grand_total = 0.0
            output_row = 2
            valid_items = 0

            # อ่านข้อมูลโดยข้ามหัวตารางและแถวว่าง
            for row_index, row in enumerate(
                input_sheet.iter_rows(
                    min_row=2, values_only=True
                ),
                start=2
            ):
                if not any(value is not None for value in row):
                    continue

                product_name = row[0] if len(row) > 0 else None
                quantity = row[1] if len(row) > 1 else None
                unit_price = row[2] if len(row) > 2 else None

                if product_name is None:
                    print(f"ข้ามแถว {row_index}: ไม่มีชื่อสินค้า")
                    continue

                try:
                    # ป้องกันข้อมูลที่แปลงเป็นตัวเลขไม่ได้
                    if isinstance(quantity, bool):
                        raise ValueError("Quantity ไม่ถูกต้อง")
                    if isinstance(unit_price, bool):
                        raise ValueError("Unit Price ไม่ถูกต้อง")

                    quantity_number = float(quantity)
                    unit_price_number = float(unit_price)

                    if not quantity_number.is_integer():
                        raise ValueError(
                            "Quantity ต้องเป็นจำนวนเต็ม"
                        )

                    quantity_number = int(quantity_number)

                    if quantity_number < 0 or unit_price_number < 0:
                        raise ValueError(
                            "จำนวนและราคาต้องไม่ติดลบ"
                        )

                except (ValueError, TypeError, OverflowError):
                    print(
                        f"Warning: ข้ามแถว {row_index} "
                        f"เนื่องจากข้อมูลตัวเลขไม่ถูกต้อง: {row}"
                    )
                    continue

                total_price = quantity_number * unit_price_number
                grand_total += total_price

                values = [
                    product_name,
                    quantity_number,
                    unit_price_number,
                    total_price
                ]

                for col_num, value in enumerate(values, start=1):
                    cell = output_sheet.cell(
                        row=output_row,
                        column=col_num,
                        value=value
                    )
                    cell.border = thin_border

                    if col_num in (3, 4):
                        cell.number_format = currency_format

                output_row += 1
                valid_items += 1

            # แสดงยอดรวมทั้งหมด
            total_row = output_row

            output_sheet.merge_cells(
                start_row=total_row,
                start_column=1,
                end_row=total_row,
                end_column=3
            )

            label_cell = output_sheet.cell(
                row=total_row,
                column=1,
                value="Grand Total:"
            )
            label_cell.font = total_font
            label_cell.fill = total_fill
            label_cell.alignment = Alignment(
                horizontal="right",
                vertical="center"
            )
            label_cell.border = thin_border

            total_cell = output_sheet.cell(
                row=total_row,
                column=4,
                value=grand_total
            )
            total_cell.font = total_font
            total_cell.fill = total_fill
            total_cell.number_format = currency_format
            total_cell.border = thin_border

            # ปรับความกว้างคอลัมน์
            for col_num in range(1, len(headers) + 1):
                column_letter = get_column_letter(col_num)
                max_length = 0

                for cell in output_sheet[column_letter]:
                    if cell.value is not None:
                        max_length = max(
                            max_length, len(str(cell.value))
                        )

                output_sheet.column_dimensions[
                    column_letter
                ].width = max_length + 2

            output_sheet.freeze_panes = "A2"
            output_sheet.auto_filter.ref = (
                f"A1:D{total_row - 1}"
                if valid_items > 0 else "A1:D1"
            )

            print(f"ประมวลผลสินค้าสำเร็จ {valid_items} รายการ")
            print(f"ยอดขายรวม: {grand_total:,.2f}")

            return self.save_workbook(
                output_wb, output_filename
            )

        finally:
            input_wb.close()

Writing spreadsheet-automator/src/excel_processor.py


In [6]:
%%writefile spreadsheet-automator/main.py

import os
from src.excel_processor import ExcelProcessor


def main():
    base_dir = os.path.dirname(os.path.abspath(__file__))

    input_file = os.path.join(
        base_dir, "data", "input_sales.xlsx"
    )
    output_file = os.path.join(
        base_dir, "data", "output_sales_report.xlsx"
    )

    if not os.path.exists(input_file):
        print(f"ไม่พบไฟล์ต้นฉบับ: {input_file}")
        return

    processor = ExcelProcessor()

    print("เริ่มประมวลผลข้อมูลยอดขาย...")
    success = processor.process_sales_data(
        input_file, output_file
    )

    if success:
        print("สร้างรายงานยอดขายสำเร็จ")
        print(f"ไฟล์รายงาน: {output_file}")
    else:
        print("ไม่สามารถสร้างรายงานยอดขายได้")


if __name__ == "__main__":
    main()

Writing spreadsheet-automator/main.py


In [7]:
%cd /content/spreadsheet-automator
!python main.py

/content/spreadsheet-automator
เริ่มประมวลผลข้อมูลยอดขาย...
อ่านไฟล์ /content/spreadsheet-automator/data/input_sales.xlsx สำเร็จ
สร้าง Workbook สำหรับ /content/spreadsheet-automator/data/output_sales_report.xlsx
ประมวลผลสินค้าสำเร็จ 5 รายการ
ยอดขายรวม: 3,203.97
บันทึกไฟล์แล้ว: /content/spreadsheet-automator/data/output_sales_report.xlsx
สร้างรายงานยอดขายสำเร็จ
ไฟล์รายงาน: /content/spreadsheet-automator/data/output_sales_report.xlsx


In [8]:
from openpyxl import load_workbook

report_path = "data/output_sales_report.xlsx"

wb = load_workbook(report_path, data_only=True)
ws = wb["Sales Report"]

for row in ws.iter_rows(values_only=True):
    print(row)

wb.close()

('Product Name', 'Quantity', 'Unit Price', 'Total Price')
('Laptop', 2, 1200.5, 2401)
('Mouse', 5, 25, 125)
('Keyboard', 3, 75.99, 227.97)
('Monitor', 1, 300, 300)
('Printer', 1, 150, 150)
('Grand Total:', None, None, 3203.97)


In [9]:
from openpyxl import load_workbook

wb = load_workbook("data/output_sales_report.xlsx")
ws = wb["Sales Report"]

print("ชื่อ Sheet:", wb.sheetnames)
print("หัวตารางตัวหนา:", ws["A1"].font.bold)
print("สีพื้นหัวตาราง:", ws["A1"].fill.fgColor.rgb)
print("รูปแบบราคา:", ws["D2"].number_format)
print("ตรึงแถว:", ws.freeze_panes)

wb.close()

ชื่อ Sheet: ['Sales Report']
หัวตารางตัวหนา: True
สีพื้นหัวตาราง: 004CAF50
รูปแบบราคา: #,##0.00
ตรึงแถว: A2


In [10]:
%%writefile .gitignore

__pycache__/
*.pyc
*.pyo
.Python/
env/
venv/
.venv/
.ipynb_checkpoints/
.vscode/
.env

# Ignore generated report
spreadsheet-automator/data/output_sales_report.xlsx

Writing .gitignore


In [11]:
%%writefile README.md

# LabX BASIC: Automating Spreadsheets (Excel & Google Sheets)

A Python project demonstrating Excel spreadsheet automation using
the openpyxl library.

## Features

- Load an existing Excel workbook.
- Read sales data from worksheets.
- Calculate total price for each product.
- Calculate the grand total of all sales.
- Skip empty rows and rows with invalid numeric data.
- Generate a new formatted Excel report.
- Apply fonts, fills, borders, number formats, and column widths.

## Project Structure

spreadsheet-automator/
- src/__init__.py
- src/excel_processor.py
- data/input_sales.xlsx
- data/output_sales_report.xlsx
- main.py
- .gitignore
- README.md

## Requirements

Python 3 and openpyxl.

Install the dependency:

    pip install openpyxl

## How to Run

1. Put input_sales.xlsx in the data directory.
2. Ensure the first worksheet contains the columns:
   Product Name, Quantity, Unit Price.
3. Run:

    python main.py

4. The generated report will be saved as:
   data/output_sales_report.xlsx

## Google Sheets

Google Sheets automation can be implemented using the Google Sheets API
and a client library such as gspread. Cloud access requires appropriate
authentication and API permissions. Full cloud integration is outside
the scope of this prototype.

## Data Integrity

The program preserves the source workbook by writing results to a
separate output file. It skips empty rows and reports rows containing
invalid numeric data instead of including them in the calculations.

Writing README.md


In [12]:
from google.colab import files

files.download("data/output_sales_report.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>